In [3]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [7]:
import os 
import numpy
import pandas as pd

import matplotlib.pyplot as plt

"""
0: Clear (No spreadF)
1: FSF
2: RSF
3: MSF
4: SSF
5: Unidentified
"""


filename = "./dataset/LISN_Labeled_SpreadF_2013_2025.xlsx"

df = pd.read_excel(filename,parse_dates=["datetime_object"])

print(df.head())

   Year  Month  Day  Hour  Minute  spreadF  Class     datetime_object  \
0  2013     12    4    17      33        0    0.0 2013-12-04 17:33:00   
1  2013     12    4    18       3        0    0.0 2013-12-04 18:03:00   
2  2013     12    4    18      18        0    0.0 2013-12-04 18:18:00   
3  2013     12    4    18      33        0    0.0 2013-12-04 18:33:00   
4  2013     12    4    18      48        0    0.0 2013-12-04 18:48:00   

   day_of_year month_name                                      LISN_location  \
0          338   december  2013_december_ngi_jica/data/JM91J_201333817330...   
1          338   december  2013_december_ngi_jica/data/JM91J_201333818030...   
2          338   december  2013_december_ngi_jica/data/JM91J_201333818180...   
3          338   december  2013_december_ngi_jica/data/JM91J_201333818330...   
4          338   december  2013_december_ngi_jica/data/JM91J_201333818480...   

                                           file_path  
0  /media/soporte/Data/li

In [11]:
conteo = df.groupby(df["datetime_object"].dt.hour).size()

print("Cantidad de ionogramas por horas")
print(conteo)

Cantidad de ionogramas por horas
datetime_object
0     1102
1     1962
2     2207
3     2206
4     2215
5     2214
6     2250
7     2122
8     2043
9     1947
10    1383
11    1230
12    1465
13    1442
14    1420
15    1448
16    1388
17    1330
18    1332
19    1346
20    1335
21    1393
22    1571
23    1829
dtype: int64


In [15]:
## convertimos todos los tipos de ionogramas SSF, FSF, ETC a 1 
df.loc[(df["spreadF"] > 0) & (df["spreadF"] != 5), "spreadF"] = 1
## convertimos los de valores 5 (UNCLEAR) a 0 
df.loc[(df["spreadF"] > 4) , "spreadF"] = 0

df.head()

,Year,Month,Day,Hour,Minute,spreadF,Class,datetime_object,day_of_year,month_name,LISN_location,file_path,hour,night_day
0,2013,12,4,17,33,0,0.0,2013-12-04 17:33:00,338,december,2013_december_ngi_jica/data/JM91J_201333817330...,/media/soporte/Data/lisn/2013_december_ngi_jic...,2013-12-04 17:00:00,2013-12-04
1,2013,12,4,18,3,0,0.0,2013-12-04 18:03:00,338,december,2013_december_ngi_jica/data/JM91J_201333818030...,/media/soporte/Data/lisn/2013_december_ngi_jic...,2013-12-04 18:00:00,2013-12-04
2,2013,12,4,18,18,0,0.0,2013-12-04 18:18:00,338,december,2013_december_ngi_jica/data/JM91J_201333818180...,/media/soporte/Data/lisn/2013_december_ngi_jic...,2013-12-04 18:00:00,2013-12-04
3,2013,12,4,18,33,0,0.0,2013-12-04 18:33:00,338,december,2013_december_ngi_jica/data/JM91J_201333818330...,/media/soporte/Data/lisn/2013_december_ngi_jic...,2013-12-04 18:00:00,2013-12-04
4,2013,12,4,18,48,0,0.0,2013-12-04 18:48:00,338,december,2013_december_ngi_jica/data/JM91J_201333818480...,/media/soporte/Data/lisn/2013_december_ngi_jic...,2013-12-04 18:00:00,2013-12-04


In [16]:
### buscamos si hay indicador de spread-f
### entre las 18:00 - hasta las 6 AM (23 <DAY> - 11 UTC <DAY + 1>)
### Se debe establecer que <DAY> -> 1  si sum{rango busqueda["spreadF"]} >1

# Día local asociado a la ventana nocturna
df["night_day"] = df["datetime_object"].dt.floor("D")

# Las mediciones entre 00:00 y 10:59 UTC pertenecen
# a la noche del día anterior (porque son hasta las 06:00 LT)
mask = df["datetime_object"].dt.hour < 11
df.loc[mask, "night_day"] -= pd.Timedelta(days=1)

mask_window = (
    (df["datetime_object"].dt.hour >= 23) |
    (df["datetime_object"].dt.hour < 11)
)

df_night = df[mask_window]


In [23]:
daily_label = (
    df_night.groupby("night_day")["spreadF"]
            .agg(["sum", "count"])
            .assign(spreadF_day=lambda x: ((x["sum"] > 1) & (x["count"] > 10)).astype(int))
            .reset_index()[["night_day", "spreadF_day"]]
)
print(daily_label)

     night_day  spreadF_day
0   2013-12-04            1
1   2013-12-05            1
2   2013-12-06            1
3   2013-12-07            1
4   2013-12-08            1
..         ...          ...
688 2025-03-18            1
689 2025-03-19            1
690 2025-03-20            1
691 2025-03-21            1
692 2025-03-22            0

[693 rows x 2 columns]


In [24]:
daily_label["time"] = daily_label["night_day"]

daily_label = daily_label[["time","spreadF_day"]]
daily_label.to_csv("./dataset/spread_f_ionogram.csv", index=False)